# Mood classification using CNN (happy / sad)

In [46]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
!nvidia-smi

Sun Feb  8 10:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             33W /   70W |    1146MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [48]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import cv2
import os
#image data generator is the package to lable the images & it will automatically lable all the images

In [49]:
img = image.load_img(r'/content/drive/MyDrive/CNN - Happy or Sad/training/happy/IMG_20221104_165434_150.jpg')

In [50]:
img

Output hidden; open in https://colab.research.google.com to view.

In [51]:
i1 = cv2.imread(r'/content/drive/MyDrive/CNN - Happy or Sad/training/happy/IMG_20221104_165434_150.jpg')

In [52]:
i1

array([[[255, 252, 231],
        [255, 252, 231],
        [255, 252, 231],
        ...,
        [220, 189, 164],
        [219, 188, 163],
        [217, 185, 162]],

       [[255, 252, 231],
        [255, 252, 231],
        [255, 252, 231],
        ...,
        [212, 182, 155],
        [214, 183, 158],
        [216, 185, 160]],

       [[255, 252, 231],
        [255, 252, 231],
        [255, 252, 231],
        ...,
        [200, 169, 138],
        [206, 174, 145],
        [209, 176, 150]],

       ...,

       [[ 14, 190, 136],
        [  5, 184, 129],
        [  0, 182, 128],
        ...,
        [238, 230, 213],
        [242, 234, 217],
        [248, 240, 223]],

       [[ 30, 202, 148],
        [ 18, 192, 138],
        [  5, 184, 129],
        ...,
        [231, 223, 206],
        [238, 230, 213],
        [250, 242, 225]],

       [[ 33, 204, 148],
        [ 23, 195, 139],
        [  8, 185, 128],
        ...,
        [222, 214, 197],
        [231, 223, 206],
        [248, 240, 223]]

In [53]:
i1.shape
# shape of your image height, weight, rgb

(1637, 1637, 3)

In [54]:
train = ImageDataGenerator(rescale = 1/255)
validataion = ImageDataGenerator(rescale = 1/255)
# to scale all the images i need to divide with 255
# we need to resize the image using 200, 200 pixel

In [55]:
train_dataset = train.flow_from_directory(r'/content/drive/MyDrive/CNN - Happy or Sad/training',
                                         target_size = (200,200),
                                         batch_size = 3,
                                         class_mode = 'binary')
validation_dataset = validataion.flow_from_directory(r'/content/drive/MyDrive/CNN - Happy or Sad/validation',
                                          target_size = (200,200),
                                          batch_size = 3,
                                          class_mode = 'binary')



Found 49 images belonging to 2 classes.
Found 0 images belonging to 2 classes.


In [56]:
train_dataset.class_indices

{'happy': 0, 'not happy': 1}

In [57]:
train_dataset.classes

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1], dtype=int32)

In [58]:
# now we are applying maxpooling

model = tf.keras.models.Sequential([ tf.keras.layers.Conv2D(16,(3,3),activation = 'relu',input_shape = (200,200,3)),
                                    tf.keras.layers.MaxPool2D(2,2), #3 filtr we applied hear
                                    #
                                    tf.keras.layers.Conv2D(32,(3,3),activation = 'relu'),
                                    tf.keras.layers.MaxPool2D(2,2),
                                    #
                                    tf.keras.layers.Conv2D(64,(3,3),activation = 'relu'),
                                    tf.keras.layers.MaxPool2D(2,2),
                                    ##
                                    tf.keras.layers.Flatten(),
                                    ##
                                    tf.keras.layers.Dense(512, activation = 'relu'),
                                    #
                                    tf.keras.layers.Dense(1,activation= 'sigmoid')
                                    ]
                                    )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [59]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    metrics=['accuracy']
)


In [60]:
model_fit = model.fit(train_dataset,epochs = 10)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 223ms/step - accuracy: 0.3308 - loss: 2.9261
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 190ms/step - accuracy: 0.6742 - loss: 0.6726
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - accuracy: 0.8278 - loss: 0.5644
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 149ms/step - accuracy: 0.6582 - loss: 0.5930
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.8518 - loss: 0.2904
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 142ms/step - accuracy: 0.9209 - loss: 0.2889
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 192ms/step - accuracy: 0.9442 - loss: 0.1381
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 148ms/step - accuracy: 0.9521 - loss: 0.1383
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 146ms/step - accuracy: 1.0000 - loss: 0.0196
Epoch 10/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 153ms/step - accuracy: 1.0000 - loss: 0.0047


In [61]:
dir_path = '/content/drive/MyDrive/CNN - Happy or Sad/testing'
for i in os.listdir(dir_path):
    print(i)

IMG_20220421_111303.jpg
IMG-20241111-WA0018.jpg
IMG_20221104_165434_150.jpg
IMG_20221111_145256_825.jpg
IMG_20251115_132549.jpg
IMG_20240907_135716.jpg
IMG_20220831_182952.jpg
IMG_20200926_170734.jpg
IMG_20220421_113232.jpg
IMG_20211129_071406.jpg
IMG_20220421_103236.jpg


In [62]:
dir_path = '/content/drive/MyDrive/CNN - Happy or Sad/testing'
for i in os.listdir(dir_path):
   img = image.load_img(dir_path+ '//'+i, target_size =(200,200))
   plt.imshow(img)
   plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [64]:
dir_path = r'/content/drive/MyDrive/CNN - Happy or Sad/testing'
for i in os.listdir(dir_path ):
    img = image.load_img(dir_path+ '//'+i, target_size = (200,200))
    plt.imshow(img)
    plt.show()

    x= image.img_to_array(img)
    x=np.expand_dims(x,axis = 0)
    images = np.vstack([x])

    val = model.predict(images)
    if val == 0:
        print( ' i am happy')
    else:
        print('i am not happy')

Output hidden; open in https://colab.research.google.com to view.

In [67]:
import gradio as gr
from PIL import Image
import numpy as np
from tensorflow.keras.models import load_model


def predict_mood(image):
    # Resize the image to (200, 200) as expected by the model
    img = image.resize((200, 200))

    # Convert the image to a numpy array
    x = np.array(img)

    # Expand dimensions to create a batch of 1 image
    x = np.expand_dims(x, axis=0)

    # Normalize the image (if the model was trained with normalized inputs)
    # The model was trained with rescale=1/200, so we should apply the same scaling here.
    x = x / 200.0

    # Make prediction
    val = model.predict(x)[0][0]

    # Interpret the prediction
    if val < 0.5:
        return "Happy"
    else:
        return "Not Happy"


In [68]:
iface = gr.Interface(
    fn=predict_mood,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=gr.Text(label="Predicted Mood"),
    title="Mood Classification (Happy/Not Happy)",
    description="Upload an image to classify if the person is happy or not happy."
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25d18384719377fd3a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
